# AutoGaze 실습 튜토리얼

이 노트북은 AutoGaze의 주요 기능을 직접 실행해 보며 이해할 수 있는 단계별 실습 자료입니다.

**다루는 내용**
1. 환경 확인 및 모델 로드
2. 비디오 로드 — 16프레임 샘플링
3. AutoGaze 인퍼런스
4. 출력 변수 상세 분석
5. 결과 시각화 (프레임 오버레이 · 히트맵)
6. 전체 프레임 처리 (`--all-frames` 청크 방식)
7. 결과 저장 및 불러오기 (JSON · NPZ)
8. `gazing_ratio` 파라미터 실험
9. (심화) SigLIP Vision Encoder 연동
10. (심화) NVILA 통합 — AutoGaze가 내장된 8B MLLM
11. CLI 스크립트로 실행하기

**사전 조건**
```bash
# 프로젝트 루트에서
source .venv/bin/activate
pip install -e ".[dev]"
huggingface-cli login   # 최초 1회
```

---
## 0. 환경 확인

In [ ]:
import sys, importlib, platform
import matplotlib
import matplotlib.font_manager as fm
import torch

# ── 한글 폰트 설정 (UserWarning: Glyph missing 방지) ──────────────
def _setup_korean_font():
    _sys = platform.system()
    if _sys == 'Darwin':                          # macOS
        matplotlib.rcParams['font.family'] = 'AppleGothic'
        _font = 'AppleGothic'
    elif _sys == 'Windows':
        matplotlib.rcParams['font.family'] = 'Malgun Gothic'
        _font = 'Malgun Gothic'
    else:                                         # Linux
        _candidates = [
            f.name for f in fm.fontManager.ttflist
            if any(k in f.name for k in ('Nanum', 'Malgun', 'Gothic', 'Batang', 'Dotum'))
        ]
        if _candidates:
            matplotlib.rcParams['font.family'] = _candidates[0]
            _font = _candidates[0]
        else:
            print("⚠  한글 폰트 없음 — 아래 명령으로 설치 후 커널 재시작")
            print("   sudo apt-get install fonts-nanum && fc-cache -fv")
            return
    matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
    print(f"[폰트] {_font}  (axes.unicode_minus=False)")

_setup_korean_font()
# ─────────────────────────────────────────────────────────────────

# Python / PyTorch 버전
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

# 사용 가능한 디바이스
if torch.cuda.is_available():
    print(f"Device : CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Device : MPS (Apple Silicon)")
else:
    print("Device : CPU")

import autogaze
import os
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
print("autogaze 패키지 로드 성공 ✓")

In [ ]:
from pathlib import Path

# ── 경로 설정 (필요 시 수정) ──────────────────────────────────────
MODEL_PATH  = "nvidia/AutoGaze"          # HuggingFace ID 또는 "weights/AutoGaze"
VIDEO_PATH  = "../assets/example_input.mp4"  # 예제 비디오
OUTPUT_DIR  = Path("../results/notebook")
# ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 비디오 파일 존재 확인
video_path = Path(VIDEO_PATH)
assert video_path.exists(), f"비디오를 찾을 수 없습니다: {video_path.resolve()}"
print(f"비디오: {video_path.resolve()}")
print(f"출력 디렉터리: {OUTPUT_DIR.resolve()}")

---
## 1. 모델 로드

In [ ]:
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor
from autogaze.utils import get_device

device = get_device()
print(f"사용 디바이스: {device}")

print(f"\n모델 로드 중: '{MODEL_PATH}' ...")
transform = AutoGazeImageProcessor.from_pretrained(MODEL_PATH)
model     = AutoGaze.from_pretrained(MODEL_PATH).to(device)
model.eval()

CHUNK_SIZE = model.config.max_num_frames        # 기본 16
NUM_TOKENS = model.config.num_vision_tokens_each_frame  # 기본 265
SCALES     = [int(s) for s in model.config.scales.split("+")]  # [32,64,112,224]

print(f"\n모델 설정")
print(f"  max_num_frames             : {CHUNK_SIZE}")
print(f"  num_vision_tokens_per_frame: {NUM_TOKENS}")
print(f"  scales                     : {SCALES}")
print(f"\n모델 로드 완료 ✓")

---
## 2. 비디오 로드 — 16프레임 샘플링

기본 모드는 비디오 전체 길이에서 `max_num_frames`개 프레임을 균등 간격으로 샘플링합니다.

In [ ]:
import av
import numpy as np
from autogaze.datasets.video_utils import (
    read_video_pyav, sample_frame_indices, process_video_frames,
    transform_video_for_pytorch,
)

# 비디오 메타데이터 확인
container = av.open(str(video_path))
stream    = container.streams.video[0]
total_frames = stream.frames
fps          = float(stream.average_rate)
duration_s   = total_frames / fps if fps > 0 else 0
container.close()

print(f"총 프레임 수  : {total_frames}")
print(f"FPS           : {fps:.2f}")
print(f"재생 시간     : {duration_s:.1f}초")
print(f"해상도        : {stream.width} × {stream.height}")
print(f"\n→ {CHUNK_SIZE}프레임 균등 샘플링")

In [ ]:
# 16프레임 샘플링 및 전처리
container = av.open(str(video_path))
indices   = sample_frame_indices(
    clip_len=CHUNK_SIZE, frame_sample_rate=1,
    seg_len=total_frames, random_sample_frame=False,
)
raw_video = read_video_pyav(container, indices)   # (T, H, W, 3) uint8
container.close()
raw_video = process_video_frames(raw_video, CHUNK_SIZE)

print(f"raw_video shape : {raw_video.shape}  dtype={raw_video.dtype}")
print(f"샘플링된 인덱스 : {indices.tolist()}")

In [ ]:
import matplotlib.pyplot as plt

# 샘플링된 프레임 시각화
T = raw_video.shape[0]
cols = min(T, 8)
rows = (T + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
axes = np.array(axes).flatten()

for t in range(T):
    axes[t].imshow(raw_video[t])
    axes[t].set_title(f"Frame {t+1}\n(idx {indices[t]})", fontsize=7)
    axes[t].axis("off")
for ax in axes[T:]:
    ax.axis("off")

plt.suptitle(f"샘플링된 {T}프레임 (원본 {total_frames}프레임에서)", fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. AutoGaze 인퍼런스

In [ ]:
# 전처리 → 모델 입력 텐서로 변환
video_input = transform_video_for_pytorch(raw_video, transform)  # (T, C, H, W)
video_input = video_input[None].to(device)                        # (1, T, C, H, W)

print(f"입력 텐서 shape: {video_input.shape}")
print(f"dtype: {video_input.dtype}")

In [ ]:
# ── 인퍼런스 파라미터 ──────────────────────────────────
GAZING_RATIO          = 0.75   # 최대 패치 비율 (0~1)
TASK_LOSS_REQUIREMENT = 0.7    # 재건 품질 임계값 (낮을수록 더 많은 패치)
# ─────────────────────────────────────────────────────

with torch.inference_mode():
    gaze_outputs = model(
        {"video": video_input},
        gazing_ratio=GAZING_RATIO,
        task_loss_requirement=TASK_LOSS_REQUIREMENT,
    )

print("인퍼런스 완료 ✓")
print(f"\n출력 키: {list(gaze_outputs.keys())}")

---
## 4. 출력 변수 상세 분석

| 변수 | 형태 | 설명 |
| --- | --- | --- |
| `gazing_pos` | `(B, N)` | 가이즈된 패치의 전역 토큰 인덱스 |
| `if_padded_gazing` | `(B, N)` bool | True = 패딩(더미) 가이즈 |
| `num_gazing_each_frame` | `(T,)` | 프레임별 가이즈 토큰 수 (패딩 포함) |
| `gazing_mask` | `list[(B, T, N_scale)]` | 스케일별 per-frame 불리언 마스크 |
| `num_vision_tokens_each_frame` | int | 프레임당 전체 토큰 수 (265) |

In [ ]:
pos         = gaze_outputs["gazing_pos"]            # (1, N)
if_padded   = gaze_outputs["if_padded_gazing"]      # (1, N)  bool
num_each    = gaze_outputs["num_gazing_each_frame"]  # (T,)
gazing_mask = gaze_outputs["gazing_mask"]            # list of (1, T, N_scale)

n_total_tokens = NUM_TOKENS * T
n_real         = int((~if_padded).sum().item())
n_padded       = int(if_padded.sum().item())

print("─" * 45)
print(f"gazing_pos shape     : {pos.shape}")
print(f"if_padded shape      : {if_padded.shape}")
print(f"num_gazing_each_frame: {num_each.tolist()}")
print("─" * 45)
print(f"전체 패치 수         : {n_total_tokens} ({T}프레임 × {NUM_TOKENS}토큰)")
print(f"실제 가이즈된 패치   : {n_real}  ({100*n_real/n_total_tokens:.1f}%)")
print(f"패딩(더미) 가이즈    : {n_padded}")
print("─" * 45)
print(f"\ngazing_mask 스케일별 shape:")
for si, scale in enumerate(SCALES):
    m = gazing_mask[si]
    n_patches_per_frame = m.shape[-1]
    grid = int(n_patches_per_frame ** 0.5)
    print(f"  scale {scale:3d}px: {tuple(m.shape)}  ({grid}×{grid} 패치 그리드)")

In [ ]:
# 프레임별 가이즈 수 막대 그래프
real_per_frame = []
offset = 0
for t, cnt in enumerate(num_each.tolist()):
    frame_pad = if_padded[0][offset: offset + cnt]
    real_per_frame.append(int((~frame_pad).sum().item()))
    offset += cnt

fig, ax = plt.subplots(figsize=(max(8, T * 0.6), 3))
bars = ax.bar(range(T), real_per_frame, color="steelblue", edgecolor="white")
ax.axhline(y=sum(real_per_frame)/T, color="tomato", linestyle="--",
           label=f"평균 {sum(real_per_frame)/T:.1f}")
ax.set_xlabel("프레임 번호 (0-based)")
ax.set_ylabel("선택된 패치 수")
ax.set_title(f"프레임별 실제 가이즈 패치 수 (gazing_ratio={GAZING_RATIO}, threshold={TASK_LOSS_REQUIREMENT})")
ax.set_xticks(range(T))
for bar, v in zip(bars, real_per_frame):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(v), ha="center", va="bottom", fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n총 선택: {n_real} / {n_total_tokens} ({100*n_real/n_total_tokens:.1f}%)")

---
## 5. 결과 시각화

### 5-A. 단일 프레임 — 스케일별 가이즈 오버레이

In [ ]:
import torch.nn.functional as F
import matplotlib.patches as mpatches
from autogaze.utils import UnNormalize

# 정규화 해제 (시각화용)
unnorm = UnNormalize(
    transform.image_mean,
    transform.image_std,
    getattr(transform, "rescale_factor", 1.0 / 255.0),
)
video_np = unnorm(transform_video_for_pytorch(raw_video, transform)).cpu().float().numpy()
# video_np: (T, C, H, W), float32 in [0, 1]

def overlay_gaze(frame_chw, mask_hw, scale, dim=0.25):
    """가이즈 마스크를 프레임에 오버레이해 uint8 HWC 이미지 반환."""
    ft = torch.from_numpy(frame_chw).unsqueeze(0)
    fs = F.interpolate(ft, size=(scale, scale), mode="bicubic", align_corners=False)
    fs = fs.squeeze().clamp(0, 1).numpy()  # C H W
    m  = F.interpolate(
        torch.from_numpy(mask_hw).unsqueeze(0).unsqueeze(0).float(),
        size=(scale, scale), mode="nearest",
    ).squeeze().numpy()
    display = fs * (dim + (1 - dim) * m[None])
    return (np.clip(display.transpose(1, 2, 0), 0, 1) * 255).astype(np.uint8)

print("시각화 유틸리티 준비 완료 ✓")

In [ ]:
# ── 시각화할 프레임 선택 ──
FRAME_IDX = 0   # 0 ~ T-1 범위에서 선택
# ─────────────────────────

fig, axes = plt.subplots(1, 1 + len(SCALES),
                          figsize=(3.5 * (1 + len(SCALES)), 3.5))

# 원본 프레임
axes[0].imshow(raw_video[FRAME_IDX])
axes[0].set_title(f"원본  Frame {FRAME_IDX+1}", fontsize=10)
axes[0].axis("off")

# 스케일별 오버레이
for si, scale in enumerate(SCALES):
    m_scale = gazing_mask[si][0]   # (T, N_scale)
    pg      = int(m_scale.shape[-1] ** 0.5)
    mask_hw = m_scale[FRAME_IDX].reshape(pg, pg).cpu().float().numpy()

    img = overlay_gaze(video_np[FRAME_IDX], mask_hw, scale)
    axes[si + 1].imshow(img)

    # 패치 경계선
    patch_px = scale // pg
    for pi in range(pg):
        for pj in range(pg):
            if mask_hw[pi, pj] > 0.5:
                axes[si + 1].add_patch(mpatches.Rectangle(
                    (pj * patch_px - 0.5, pi * patch_px - 0.5),
                    patch_px, patch_px,
                    linewidth=0.8, edgecolor="red", facecolor="none",
                ))
    axes[si + 1].set_title(f"Scale {scale}px\n({int(mask_hw.sum())} 패치)", fontsize=9)
    axes[si + 1].axis("off")

plt.suptitle(f"Frame {FRAME_IDX+1} 가이즈 오버레이 (스케일별)", fontsize=12)
plt.tight_layout()
plt.show()

### 5-B. 전체 프레임 그리드 — scale224 오버레이

In [ ]:
largest_si    = len(SCALES) - 1
largest_scale = SCALES[largest_si]
largest_mask  = gazing_mask[largest_si][0]  # (T, N)
largest_pg    = int(largest_mask.shape[-1] ** 0.5)

cols = min(T, 8)
rows = (T + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
axes = np.array(axes).flatten()

for t in range(T):
    mask_hw = largest_mask[t].reshape(largest_pg, largest_pg).cpu().float().numpy()
    img = overlay_gaze(video_np[t], mask_hw, largest_scale)
    axes[t].imshow(img)
    axes[t].set_title(f"F{t+1}  ({int(mask_hw.sum())}p)", fontsize=7)
    axes[t].axis("off")
for ax in axes[T:]:
    ax.axis("off")

plt.suptitle(f"전체 {T}프레임  Scale-{largest_scale} 가이즈 오버레이", fontsize=12)
plt.tight_layout()
plt.show()

### 5-C. 멀티스케일 히트맵

In [ ]:
RENDER = 224

cols = min(T, 8)
rows = (T + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
axes = np.array(axes).flatten()

for t in range(T):
    # 모든 스케일 마스크를 RENDER 크기로 업샘플해 합산
    heat = np.zeros((RENDER, RENDER), dtype=np.float32)
    for si in range(len(SCALES)):
        ms = gazing_mask[si][0][t]
        pg = int(ms.shape[-1] ** 0.5)
        m_up = F.interpolate(
            ms.reshape(pg, pg).cpu().float().unsqueeze(0).unsqueeze(0),
            size=(RENDER, RENDER), mode="nearest",
        ).squeeze().numpy()
        heat += m_up

    heat = heat / heat.max() if heat.max() > 0 else heat

    # 원본과 블렌드
    orig_t = torch.from_numpy(raw_video[t]).permute(2, 0, 1).float() / 255.0
    orig_r = F.interpolate(orig_t.unsqueeze(0), size=(RENDER, RENDER),
                           mode="bicubic", align_corners=False).squeeze().permute(1,2,0).numpy()
    cmap = plt.get_cmap("hot")
    heat_rgb   = cmap(heat)[:, :, :3].astype(np.float32)
    alpha      = 0.55
    blended    = np.clip(
        orig_r * (1 - alpha * heat[:, :, None]) + heat_rgb * alpha * heat[:, :, None],
        0, 1,
    )

    axes[t].imshow(blended)
    axes[t].set_title(f"F{t+1}", fontsize=7)
    axes[t].axis("off")

for ax in axes[T:]:
    ax.axis("off")

plt.suptitle("멀티스케일 가이즈 히트맵 (전체 프레임)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. 전체 프레임 처리 (`--all-frames` 청크 방식)

비디오를 16프레임짜리 청크로 나눠 각각 AutoGaze를 실행한 뒤,  
결과를 하나의 `gazing_mask` / `gazing_pos`로 합칩니다.

In [ ]:
# autogaze/infer.py에 구현된 함수 직접 활용
import sys; sys.path.insert(0, "..")
from autogaze.infer import load_all_frames, run_inference_chunked

print(f"전체 프레임 로드 중: {video_path.name} ...")
all_frames = load_all_frames(video_path)   # (T_total, H, W, 3) uint8
T_total    = len(all_frames)
n_chunks   = (T_total + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"총 프레임 수  : {T_total}")
print(f"청크 크기     : {CHUNK_SIZE}")
print(f"청크 수       : {n_chunks}")

In [ ]:
print("청크별 인퍼런스 실행 중 ...")
merged_outputs = run_inference_chunked(
    model, all_frames, transform, device,
    chunk_size=CHUNK_SIZE,
    gazing_ratio=GAZING_RATIO,
    task_loss_req=TASK_LOSS_REQUIREMENT,
)

T_all    = len(merged_outputs["num_gazing_each_frame"])
n_real_all = int((~merged_outputs["if_padded_gazing"]).sum().item())
n_total_all = NUM_TOKENS * T_all

print(f"\n병합 결과")
print(f"  처리된 프레임 수: {T_all}")
print(f"  gazing_mask[3] shape: {tuple(merged_outputs['gazing_mask'][3].shape)}")
print(f"  실제 가이즈: {n_real_all} / {n_total_all} ({100*n_real_all/n_total_all:.1f}%)")
print(f"  프레임별: {merged_outputs['num_gazing_each_frame'].tolist()}")

In [ ]:
# 전체 프레임 가이즈 수 시각화
num_each_all = merged_outputs["num_gazing_each_frame"]
if_pad_all   = merged_outputs["if_padded_gazing"]

real_all = []
offset = 0
for cnt in num_each_all.tolist():
    pad = if_pad_all[0][offset: offset + cnt]
    real_all.append(int((~pad).sum().item()))
    offset += cnt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 왼쪽: 16프레임 샘플링
axes[0].bar(range(T), real_per_frame, color="steelblue", edgecolor="white")
axes[0].set_title(f"16프레임 샘플링 (총 {sum(real_per_frame)}패치)")
axes[0].set_xlabel("프레임 번호")
axes[0].set_ylabel("가이즈 패치 수")

# 오른쪽: 전체 프레임
chunk_colors = ["steelblue", "tomato", "seagreen", "darkorange",
                "mediumpurple", "goldenrod", "teal", "coral"]
for t, v in enumerate(real_all):
    c = chunk_colors[(t // CHUNK_SIZE) % len(chunk_colors)]
    axes[1].bar(t, v, color=c, edgecolor="white")
axes[1].set_title(f"전체 {T_all}프레임 처리 (총 {sum(real_all)}패치, 색상=청크)")
axes[1].set_xlabel("프레임 번호")

# 청크 경계선
for c in range(1, n_chunks):
    axes[1].axvline(x=c * CHUNK_SIZE - 0.5, color="black", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 전체 프레임 Scale-224 오버레이 그리드
all_video_np = UnNormalize(
    transform.image_mean, transform.image_std,
    getattr(transform, "rescale_factor", 1/255.0),
)(
    transform_video_for_pytorch(all_frames, transform)
).cpu().float().numpy()

m_all    = merged_outputs["gazing_mask"][largest_si][0]  # (T_all, N)
pg_all   = int(m_all.shape[-1] ** 0.5)
scale_px = largest_scale

cols = min(T_all, 8)
rows = (T_all + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
axes = np.array(axes).flatten()

for t in range(T_all):
    mask_hw = m_all[t].reshape(pg_all, pg_all).cpu().float().numpy()
    img = overlay_gaze(all_video_np[t], mask_hw, scale_px)
    axes[t].imshow(img)
    axes[t].set_title(f"F{t+1}\n({int(mask_hw.sum())}p)", fontsize=6)
    axes[t].axis("off")
for ax in axes[T_all:]:
    ax.axis("off")

plt.suptitle(f"전체 {T_all}프레임 Scale-{scale_px} 오버레이", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. 결과 저장 및 불러오기

### 7-A. NPZ (압축 numpy 배열) 저장

In [ ]:
from autogaze.infer import save_npy

npz_path = save_npy(gaze_outputs, OUTPUT_DIR, video_path)
print(f"NPZ 저장: {npz_path}")
print(f"파일 크기: {npz_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# NPZ 불러오기 및 내용 확인
data = np.load(npz_path)
print(f"저장된 키: {data.files}")
print()

for key in data.files:
    arr = data[key]
    if arr.ndim == 0:
        print(f"  {key}: {arr.item()}")
    else:
        summary = f"shape={arr.shape}, dtype={arr.dtype}"
        if arr.dtype == bool or arr.dtype == np.bool_:
            summary += f", True 수={arr.sum()}"
        elif arr.ndim >= 2:
            summary += f", sum/frame={arr.sum(axis=-1).tolist()}"
        print(f"  {key}: {summary}")

In [ ]:
# 스케일별 가이즈 패치 수 분석
print("스케일별 가이즈 패치 수 (프레임별)")
print("─" * 50)
print(f"{'Scale':>8} | {'프레임별 가이즈 패치 수 (합계)':}")
print("─" * 50)
for scale in SCALES:
    m = data[f"scale_{scale}"]  # (T, N_patches)
    per_frame = m.sum(axis=-1).astype(int)
    total     = per_frame.sum()
    print(f"{scale:>6}px | {per_frame.tolist()}  (총 {total})")

### 7-B. JSON (NTP 학습 레이블) 저장

In [ ]:
import json
from autogaze.infer import save_json

all_json: dict = {}
save_json(gaze_outputs, video_path, all_json)

json_path = OUTPUT_DIR / "gazing_labels.json"
with open(json_path, "w") as f:
    json.dump(all_json, f, indent=2)

print(f"JSON 저장: {json_path}")
print()

# 포맷 확인
key = list(all_json.keys())[0]
entry = all_json[key]
print(f"키: {key}")
print(f"프레임 수: {len(entry['gazing_pos'])}")
print()
for t, (pos_list, loss_list) in enumerate(zip(entry["gazing_pos"], entry["task_losses"])):
    print(f"  Frame {t+1:2d}: gazing_pos={pos_list[:5]}{'...' if len(pos_list)>5 else ''}  ({len(pos_list)} 패치)")

### 7-C. 시각화 PNG 및 MP4 저장

In [ ]:
from autogaze.infer import save_viz, save_frames, save_video

viz_kwargs = dict(
    transform=transform,
    normalize_mean=transform.image_mean,
    normalize_std=transform.image_std,
    normalize_rescale=getattr(transform, "rescale_factor", 1/255.0),
)

# 전체 그리드 PNG
p_viz = save_viz(gaze_outputs, raw_video, OUTPUT_DIR, video_path, **viz_kwargs)
print(f"viz PNG : {p_viz}")

# 프레임별 PNG
p_frames = save_frames(gaze_outputs, raw_video, OUTPUT_DIR, video_path, **viz_kwargs)
print(f"frames  : {p_frames[0].parent}/  ({len(p_frames)} 파일)")

# MP4 비디오
p_video = save_video(gaze_outputs, raw_video, OUTPUT_DIR, video_path,
                     fps=4.0, **viz_kwargs)
print(f"video   : {p_video}")

In [ ]:
# 저장된 PNG 그리드 인라인 표시
from PIL import Image

img = Image.open(p_viz)
w, h = img.size
scale_factor = min(1.0, 1400 / w)
display_size = (int(w * scale_factor), int(h * scale_factor))

fig, ax = plt.subplots(figsize=(display_size[0]/100, display_size[1]/100))
ax.imshow(np.array(img.resize(display_size, Image.LANCZOS)))
ax.axis("off")
ax.set_title(f"저장된 viz PNG ({w}×{h}px)", fontsize=10)
plt.tight_layout()
plt.show()

---
## 8. gazing_ratio 파라미터 실험

`gazing_ratio`와 `task_loss_requirement` 값을 바꾸면 선택 패치 수가 어떻게 달라지는지 비교합니다.

In [ ]:
ratios = [0.1, 0.25, 0.5, 0.75, 1.0]
results_by_ratio = {}

for ratio in ratios:
    with torch.inference_mode():
        out = model(
            {"video": video_input},
            gazing_ratio=ratio,
            task_loss_requirement=None,   # 임계값 없이 비율만 제어
        )
    n = int((~out["if_padded_gazing"]).sum().item())
    results_by_ratio[ratio] = n
    print(f"  gazing_ratio={ratio:.2f} → 선택 패치 수 {n:4d} / {n_total_tokens} ({100*n/n_total_tokens:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
rs = list(results_by_ratio.keys())
ns = list(results_by_ratio.values())
ax.plot(rs, ns, marker="o", color="steelblue", linewidth=2)
ax.axhline(y=n_total_tokens, color="gray", linestyle="--", alpha=0.5, label="전체 패치")
ax.set_xlabel("gazing_ratio")
ax.set_ylabel("선택된 패치 수")
ax.set_title("gazing_ratio에 따른 선택 패치 수 (task_loss_requirement=None)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# task_loss_requirement 영향 실험
thresholds = [0.3, 0.5, 0.7, 0.9, None]  # None = 임계값 없음
results_by_threshold = {}

for thr in thresholds:
    with torch.inference_mode():
        out = model(
            {"video": video_input},
            gazing_ratio=0.75,
            task_loss_requirement=thr,
        )
    n = int((~out["if_padded_gazing"]).sum().item())
    label = str(thr) if thr is not None else "None (비활성)"
    results_by_threshold[label] = n
    print(f"  task_loss_requirement={label:15s} → 패치 수 {n:4d} ({100*n/n_total_tokens:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
labels = list(results_by_threshold.keys())
values = list(results_by_threshold.values())
bars = ax.bar(labels, values, color="tomato", edgecolor="white")
ax.axhline(y=n_total_tokens, color="gray", linestyle="--", alpha=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(v), ha="center", fontsize=9)
ax.set_xlabel("task_loss_requirement")
ax.set_ylabel("선택된 패치 수")
ax.set_title("task_loss_requirement에 따른 선택 패치 수 (gazing_ratio=0.75)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. (심화) SigLIP Vision Encoder 연동

AutoGaze가 선택한 패치만 SigLIP에 전달해 효율적으로 인코딩합니다.

In [ ]:
try:
    from transformers import AutoImageProcessor
    from autogaze.vision_encoders.siglip import SiglipVisionModel

    SIGLIP_MODEL = "google/siglip2-base-patch16-224"
    print(f"SigLIP 로드 중: {SIGLIP_MODEL} ...")

    siglip_transform = AutoImageProcessor.from_pretrained(SIGLIP_MODEL)
    siglip_model = SiglipVisionModel.from_pretrained(
        SIGLIP_MODEL,
        scales=model.config.scales,   # "32+64+112+224"
        attn_implementation="sdpa",
    ).to(device).eval()

    print("SigLIP 로드 완료 ✓")
    SIGLIP_AVAILABLE = True

except Exception as e:
    print(f"SigLIP 로드 실패 (스킵): {e}")
    SIGLIP_AVAILABLE = False

In [ ]:
if SIGLIP_AVAILABLE:
    # SigLIP 전처리
    video_siglip = transform_video_for_pytorch(raw_video, siglip_transform)
    video_siglip = video_siglip[None].to(device)  # (1, T, C, H, W)

    # 가이즈된 패치만 인코딩
    with torch.inference_mode():
        siglip_out = siglip_model(video_siglip, gazing_info=gaze_outputs)

    last_hs = siglip_out.last_hidden_state  # (1, N_gazed, D)
    print(f"SigLIP 출력 shape (패딩 포함): {tuple(last_hs.shape)}")

    # 패딩 제거 → 배치별 가변 길이 피처
    features = [
        feat[~pad]
        for feat, pad in zip(last_hs, gaze_outputs["if_padded_gazing"])
    ]
    print(f"패딩 제거 후 피처 shape: {tuple(features[0].shape)}")
    print(f"  → {features[0].shape[0]}개 가이즈 토큰 × {features[0].shape[1]}차원 임베딩")

    # 효율성 비교
    total_patches = NUM_TOKENS * T
    gazed_patches = features[0].shape[0]
    print(f"\n효율성")
    print(f"  전체 패치 처리 시 토큰 수 : {total_patches}")
    print(f"  AutoGaze 선택 토큰 수     : {gazed_patches}")
    print(f"  절감률                    : {100*(1 - gazed_patches/total_patches):.1f}%")

---
## 10. (심화) NVILA 통합 — AutoGaze가 내장된 8B MLLM

**NVILA-8B-HD-Video**는 AutoGaze가 내장된 멀티모달 언어 모델입니다.  
AutoGaze를 별도로 실행하지 않아도 비디오 질의응답을 바로 수행합니다.

```
비디오 입력 (최대 4K·1K 프레임)
    │
    ├── [AutoGaze 3M]    ← 정보 패치 자동 선택 (내장)
    │        │ gazing_pos
    ├── [SigLIP / ViT]   ← 선택된 패치만 인코딩  (scales: 56+112+196+392)
    │        │ vision_tokens
    ├── [Projector]      ← TokenShuffle(9) + MLP
    │        │
    └── [Qwen2-7B]       ← 질의응답 생성
```

**NVILA 특이사항 (독립 실행 시)**
- 텍스트에 비디오 토큰 플레이스홀더 필수: `<vila/video>\n질문`
- `processing_nvila.py` 패치 필요 (가이드 참조):
  - `"cuda"` 하드코딩 → `cuda/mps/cpu` 자동 감지
  - `num_video_frames: 8` → `16` (AutoGaze max_num_frames 배수 요건)
- 완성된 테스트 스크립트: `scripts/test_nvila.py`

> **사전 요건**: `bash scripts/download_models.sh weights nvila` (~16 GB)  
> **메모리**: CUDA ≥ 20 GB VRAM 또는 MPS ≥ 24 GB Unified Memory

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

NVILA_PATH   = Path("../weights/NVILA-8B-HD-Video")
AUTOGAZE_PATH = Path("../weights/AutoGaze")

# ── 재귀 디바이스 이동 헬퍼 ────────────────────────────────────────
def _to_device_nvila(v, dev):
    if isinstance(v, torch.Tensor):
        return v.to(dev)
    if isinstance(v, list):
        return [_to_device_nvila(x, dev) for x in v]
    if isinstance(v, dict):
        return {k: _to_device_nvila(vv, dev) for k, vv in v.items()}
    return v

# ── NVILA 로드 ──────────────────────────────────────────────────────
NVILA_AVAILABLE = False

if not NVILA_PATH.exists():
    print(f"NVILA 가중치 없음: {NVILA_PATH}")
    print("  → bash scripts/download_models.sh weights nvila  (~16 GB)")
else:
    try:
        from transformers import AutoProcessor, AutoModel

        print(f"NVILA 프로세서 로드 중 ...")
        nvila_processor = AutoProcessor.from_pretrained(
            str(NVILA_PATH),
            trust_remote_code=True,
            autogaze_model_id=str(AUTOGAZE_PATH),  # 로컬 AutoGaze 사용
        )
        # num_video_frames를 AutoGaze 요건(16 배수)에 맞게 설정
        nvila_processor.num_video_frames = 16

        print(f"NVILA 모델 로드 중 (~16 GB, 수 분 소요) ...")
        nvila_model = AutoModel.from_pretrained(
            str(NVILA_PATH),
            trust_remote_code=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        nvila_model.eval()

        n_params = sum(p.numel() for p in nvila_model.parameters()) / 1e9
        print(f"NVILA 로드 완료 ✓  ({n_params:.2f}B 파라미터)")
        print(f"  백본  : Qwen2-7B")
        print(f"  비전  : SigLIP (scales={nvila_processor.target_scales})")
        print(f"  프레임: {nvila_processor.num_video_frames} tile + {nvila_processor.num_video_frames_thumbnail} thumbnail")
        NVILA_AVAILABLE = True

    except Exception as e:
        print(f"NVILA 로드 실패: {type(e).__name__}: {e}")
        print()
        print("해결 방법:")
        print("  1. bash scripts/download_models.sh weights nvila")
        print("  2. pip install opencv-python-headless einops accelerate")
        print("  3. processing_nvila.py 패치 확인 (Section 10 마크다운 참조)")

In [ ]:
if NVILA_AVAILABLE:
    import time as _time

    # NVILA 비디오 토큰 플레이스홀더 — 반드시 텍스트에 포함해야 합니다
    video_token = nvila_processor.tokenizer.video_token  # '<vila/video>'
    question    = "이 비디오에서 무엇이 일어나고 있나요? 주요 장면을 설명해 주세요."
    prompt      = f"{video_token}\n{question}"

    print(f"비디오 토큰 : {repr(video_token)}")
    print(f"질문        : {question}")
    print(f"입력 프레임 : {T} (tile용)  → NVILA 내부에서 AutoGaze로 패치 선택")
    print()

    # ── 전처리: 비디오 경로를 직접 넘기면 프레임 추출 + AutoGaze 실행 ──
    t0 = _time.perf_counter()
    inputs = nvila_processor(text=prompt, videos=str(video_path))
    t_prep = _time.perf_counter() - t0

    # ── device 이동 + list → tensor 변환 ────────────────────────────
    model_device = next(nvila_model.parameters()).device
    inputs_dev = _to_device_nvila(dict(inputs), model_device)
    for key in ("input_ids", "attention_mask"):
        if key in inputs_dev and isinstance(inputs_dev[key], list):
            inputs_dev[key] = torch.tensor(inputs_dev[key], device=model_device)

    input_ids   = inputs_dev.pop("input_ids")
    extra_kwargs = inputs_dev

    # ── 생성 ─────────────────────────────────────────────────────────
    t0 = _time.perf_counter()
    with torch.inference_mode():
        generated_ids = nvila_model.generate(
            input_ids=input_ids,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            top_p=None,
            **extra_kwargs,
        )
    t_gen = _time.perf_counter() - t0

    new_ids = generated_ids[:, input_ids.shape[1]:]
    answer  = nvila_processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
    n_tok   = new_ids.shape[1]

    print(f"[NVILA 답변]\n{answer}")
    print()
    print(f"  전처리: {t_prep:.1f}s  |  생성: {t_gen:.1f}s  |  {n_tok}토큰  |  {n_tok/max(t_gen,0.001):.1f} tok/s")
    print()
    print("개별 AutoGaze vs NVILA 비교:")
    print(f"  개별 AutoGaze : gazing_pos 출력 → SigLIP 별도 호출 필요 (scales: 32+64+112+224)")
    print(f"  NVILA 통합    : .generate() 한 번으로 완결 (scales: 56+112+196+392, 더 높은 해상도)")

else:
    print("NVILA를 로드하지 못했습니다.")
    print("weights/NVILA-8B-HD-Video 가 있고 메모리가 충분한 환경에서 실행하세요.")
    print("CLI 스크립트로도 실행 가능: python scripts/test_nvila.py")

In [ ]:
import subprocess, shlex

def run_cmd(cmd):
    """명령을 실행하고 출력을 표시합니다."""
    print(f"$ {cmd}\n")
    result = subprocess.run(
        shlex.split(cmd), capture_output=True, text=True, cwd=".."
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"[STDERR] {result.stderr}")
    return result.returncode

# 예제: 16프레임 샘플링으로 모든 포맷 생성
run_cmd(
    f"python -m autogaze.infer assets/example_input.mp4 "
    f"--output-dir {OUTPUT_DIR} "
    f"--output-format frames,video,npy"
)

In [ ]:
# 예제: 전체 프레임 처리 (--all-frames)
run_cmd(
    f"python -m autogaze.infer assets/example_input.mp4 "
    f"--all-frames "
    f"--output-dir {OUTPUT_DIR}/all_frames "
    f"--output-format frames,video"
)

In [ ]:
# 출력 파일 목록 확인
print("생성된 파일 목록:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(OUTPUT_DIR)}  ({size_kb:.1f} KB)")

---
## 정리

| 단계 | 핵심 함수 / 클래스 | 비고 |
| --- | --- | --- |
| 모델 로드 | `AutoGaze.from_pretrained()` | HF ID 또는 로컬 경로 |
| 전처리 | `AutoGazeImageProcessor`, `transform_video_for_pytorch()` | 224×224 리사이즈 |
| 인퍼런스 | `model({"video": tensor}, gazing_ratio, task_loss_requirement)` | `torch.inference_mode()` 필수 |
| 전체 프레임 | `load_all_frames()` + `run_inference_chunked()` | 16프레임 청크로 분할 |
| 결과 저장 | `save_npy()`, `save_json()`, `save_viz()`, `save_video()` | `autogaze.infer` 모듈 |
| SigLIP 연동 | `SiglipVisionModel(gazing_info=gaze_outputs)` | `if_padded_gazing`으로 패딩 제거 |

**다음 단계**
- `GUIDE_KO.md` → 학습 파이프라인 (Stage 1 NTP / Stage 2 RL)
- `scripts/train_ntp_single_gpu.sh` → 소규모 NTP 학습 테스트
- `scripts/train_rl_single_gpu.sh` → GRPO RL 후학습 테스트